# 01 · Data cleaning


Goal: find the data-quality problems in the 17 raw CSVs, fix them, and save clean tables to `Data_clean/`. The fixes live in `src/clean.py` so the notebook and `run_all.py` give the same result.


In [1]:
# Setup: works from the project root or from notebooks/
import sys
from pathlib import Path
BASE = Path.cwd() if (Path.cwd() / "Data").exists() else Path.cwd().parent
sys.path.append(str(BASE / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

from config import *
from theme import *
apply_theme()
make_dirs()
pd.set_option("display.max_columns", 40)
pd.set_option("display.float_format", "{:,.3f}".format)
from load import load_raw
import clean as CL

## 1. Load the raw data

In [2]:
raw = load_raw()
pd.DataFrame({name: [len(df), df.shape[1]] for name, df in raw.items()}, index=["rows", "columns"]).T

,rows,columns
bookings,306798,19
calendar,1826,13
channels,7,5
cities,15,7
competitor_rates,24246,6
experiments,4,7
fx_rates,180,3
guests,234675,11
hotels,15,11
local_events,499,9


## 2. Look for problems
### Bookings

In [3]:
b = raw["bookings"]
checks = {
    "exact duplicate rows": b.duplicated().sum(),
    "duplicate booking_id": b.booking_id.duplicated().sum(),
    "checkout before checkin": (b.checkout_date < b.checkin_date).sum(),
    "adr_eur = 0": (b.adr_eur <= 0).sum(),
    "adr_eur > 3000 (cents?)": (b.adr_eur > 3000).sum(),
    "adults = 0": (b.adults == 0).sum(),
    "missing values": b.drop(columns="status_datetime").isna().sum().sum(),
}
pd.Series(checks, name="rows")

exact duplicate rows       1500
duplicate booking_id       1500
checkout before checkin     121
adr_eur = 0                 260
adr_eur > 3000 (cents?)      46
adults = 0                   70
missing values                0
Name: rows, dtype: int64

In [4]:
b.market_segment.value_counts()

market_segment
Leisure      184223
Business      63615
Family        35099
Group         17728
LEISURE        1277
leisure        1264
Leisure        1226
Business        448
BUSINESS        442
business        417
Family          248
family          241
FAMILY          229
Group           122
GROUP           111
group           108
Name: count, dtype: int64

In [5]:
# nights should equal checkout - checkin; on swapped rows it equals checkin - checkout
sw = b[b.checkout_date < b.checkin_date]
(sw.checkin_date - sw.checkout_date).dt.days.eq(sw.nights).mean()

np.float64(1.0)

### Guests

In [6]:
g = raw["guests"]
print("country values that are not 2-letter upper case:", (~g.home_country.astype(str).str.fullmatch(r"[A-Z]{2}")).sum())
print(g.home_country[~g.home_country.astype(str).str.fullmatch(r"[A-Z]{2}")].value_counts().head(8))
print("\nbirth_year missing:", g.birth_year.isna().sum())
print("impossible birth_year:", (~g.birth_year.between(1930, 2007) & g.birth_year.notna()).sum(), sorted(g.birth_year[~g.birth_year.between(1930, 2007)].dropna().unique()))
print("email_hash shared by 2+ profiles:", g.email_hash.duplicated().sum())

country values that are not 2-letter upper case: 7048
home_country
Germany           620
de                403
United Kingdom    304
Poland            279
Latvia            263
France            240
Netherlands       214
USA               213
Name: count, dtype: int64

birth_year missing: 16431
impossible birth_year: 140 [np.float64(1111.0), np.float64(1900.0), np.float64(1901.0), np.float64(2024.0), np.float64(2025.0)]
email_hash shared by 2+ profiles: 2782


### Reviews, competitor rates, stay charges, quotes

In [7]:
r = raw["reviews"]
r.groupby([r.review_datetime.dt.year, "source"]).overall_score.agg(["mean", "max"]).unstack().round(2)

mean                                 max                   \
source          BookNest Post-stay survey TripGlobe BookNest Post-stay survey   
review_datetime                                                                 
2021               8.260            8.240     4.130   10.000           10.000   
2022               7.120            7.090     3.570   10.000           10.000   
2023               7.910            7.890     7.940   10.000           10.000   
2024               7.980            8.000     8.020   10.000           10.000   
2025               8.030            8.010     8.040   10.000           10.000   
2026               7.960            7.860     8.010   10.000           10.000   

                           
source          TripGlobe  
review_datetime            
2021                5.000  
2022                5.000  
2023               10.000  
2024               10.000  
2025               10.000  
2026               10.000

TripGlobe scores before 2023 never go above 5: that source used a 1-5 scale and must be doubled.

In [8]:
cr = raw["competitor_rates"]
print("currencies:", cr.currency.value_counts().to_dict())
print("missing rates:", cr.comp_avg_rate.isna().sum())
sc = raw["stay_charges"]
print("charge currencies:", sc.currency.value_counts().to_dict())
print("negative charges:", (sc.amount < 0).sum(), sc[sc.amount < 0].category.unique())

currencies: {'EUR': 17397, 'PLN': 3197, 'CZK': 1826, 'DKK': 1826}
missing rates: 487
charge currencies: {'EUR': 300224, 'PLN': 58015, 'CZK': 32896, 'DKK': 27151}
negative charges: 1894 <ArrowStringArray>
['Adjustment']
Length: 1, dtype: str


In [9]:
q = raw["rate_quotes"]
s = q.groupby("session_id").agg(quotes=("quote_id", "size"), first=("quote_datetime", "min"), last=("quote_datetime", "max"), conv=("converted", "sum"))
s["sec_per_quote"] = (s["last"] - s["first"]).dt.total_seconds() / s.quotes
print(s.quotes.describe().round(1))
s.sort_values("quotes", ascending=False).head(8)

count   335,370.000
mean          1.400
std           3.300
min           1.000
25%           1.000
50%           1.000
75%           2.000
max         505.000
Name: quotes, dtype: float64


,quotes,first,last,conv,sec_per_quote
session_id,,,,,
S0335356,505,2023-06-08 00:34:07,2023-06-08 01:42:20,0,8.105
S0335369,495,2021-08-13 14:58:48,2021-08-13 16:02:55,0,7.772
S0335362,483,2025-10-10 05:10:13,2025-10-10 06:15:43,0,8.137
S0335347,473,2022-02-15 12:16:50,2022-02-15 13:20:19,0,8.053
S0335359,464,2024-03-22 06:35:42,2024-03-22 07:38:19,0,8.097
S0335346,457,2023-05-06 06:33:45,2023-05-06 07:32:03,0,7.654
S0335357,442,2021-05-20 15:11:02,2021-05-20 16:11:23,0,8.192
S0335360,439,2024-10-15 12:49:28,2024-10-15 13:50:21,0,8.321


A handful of sessions show hundreds of quotes a few seconds apart and never book: these are bots / scrapers.

## 3. Clean everything
`clean_all()` runs every fix, prints one line per step and saves the tables to `Data_clean/`.

In [10]:
clean, log = CL.clean_all()
log

Cleaning ...
  guests            home_country standardised to ISO-2 upper case                             7,048
  guests            impossible birth_year set to missing                                        140
  guests            birth_year missing (left missing)                                        16,571


  guests            duplicate profiles merged (same email_hash)                               2,782


  bookings          exact duplicate rows removed                                              1,500
  bookings          guest_id remapped to merged profile                                       4,250
  bookings          market_segment trimmed / title case                                       6,106
  bookings          checkin/checkout swapped back                                               120
  bookings          adults = 0 set to 1                                                          70
  bookings          adr_eur entered in cents divided by 100                                      45


  bookings          adr_eur = 0 imputed (room type x plan x month median)                       260


  stay_charges      local currency converted to EUR with fx_rates                           118,062
  stay_charges      negative Adjustment rows kept (flag is_adjustment)                        1,894
  competitor_rates  local currency converted to EUR with fx_rates                             6,849
  competitor_rates  10x spikes set to missing                                                    60
  competitor_rates  missing rates interpolated per hotel                                        547
  reviews           TripGlobe 2021-2022 scores rescaled from 1-5 to 1-10                      7,954


  reviews           review_datetime before stay replaced by checkout + median delay             300


  rate_quotes       bot sessions removed (50+ quotes, < 30 s apart)                           9,457
  staffing_monthly  missing hotel-months added and interpolated                                  12
  local_events      duplicate events removed                                                      8


Saved 17 clean tables to Data_clean


,table,step,rows_affected
0,guests,home_country standardised to ISO-2 upper case,7048
1,guests,impossible birth_year set to missing,140
2,guests,birth_year missing (left missing),16571
3,guests,duplicate profiles merged (same email_hash),2782
4,bookings,exact duplicate rows removed,1500
5,bookings,guest_id remapped to merged profile,4250
6,bookings,market_segment trimmed / title case,6106
7,bookings,checkin/checkout swapped back,120
8,bookings,adults = 0 set to 1,70
9,bookings,adr_eur entered in cents divided by 100,45


## 4. Check the result

In [11]:
cb = clean["bookings"]
pd.Series({
    "duplicate booking_id": cb.booking_id.duplicated().sum(),
    "checkout before checkin": (cb.checkout_date < cb.checkin_date).sum(),
    "adr <= 0 or > 3000": ((cb.adr_eur <= 0) | (cb.adr_eur > 3000)).sum(),
    "unknown guest_id": (~cb.guest_id.isin(clean["guests"].guest_id)).sum(),
    "segments": ", ".join(sorted(cb.market_segment.unique())),
})

duplicate booking_id                                      0
checkout before checkin                                   0
adr <= 0 or > 3000                                        0
unknown guest_id                                          0
segments                   Business, Family, Group, Leisure
dtype: object

In [12]:
# KPIs by arrival year from the clean data
kpi = cb.groupby("year").agg(bookings=("booking_id", "size"), cancel_rate=("is_cancelled", "mean"),
                             room_nights=("room_nights", "sum"), room_revenue_eur_m=("room_revenue_eur", lambda s: s.sum() / 1e6))
kpi

,bookings,cancel_rate,room_nights,room_revenue_eur_m
year,,,,
2021,32136,0.300,58104,7.766
2022,58376,0.230,114118,17.857
2023,64775,0.193,132138,23.099
2024,73538,0.161,156070,30.116
2025,76473,0.156,161728,32.646


In [13]:
log.to_csv(TAB_DIR / "cleaning_log.csv", index=False)
print("Saved", TAB_DIR / "cleaning_log.csv")

Saved output/tables/cleaning_log.csv
